## Notebook para validações rápidas\

### validando preprocessamento

In [1]:
import pandas as pd

In [2]:

df = pd.read_csv('data/processed/estado_nutricional_clean.csv')

display(df.info())
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1533353 entries, 0 to 1533352
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   NU_IDADE_ANO  1533353 non-null  int64  
 1   DS_FASE_VIDA  1533353 non-null  int64  
 2   SG_SEXO       1533353 non-null  int64  
 3   NU_PESO       1533353 non-null  float64
 4   NU_ALTURA     1533353 non-null  float64
 5   DS_IMC        1533353 non-null  float64
 6   ESTADO_NUTRI  1533353 non-null  object 
 7   TARGET        1533353 non-null  int64  
 8   PERC_GORDURA  1533353 non-null  float64
dtypes: float64(4), int64(4), object(1)
memory usage: 105.3+ MB


None

,NU_IDADE_ANO,DS_FASE_VIDA,SG_SEXO,NU_PESO,NU_ALTURA,DS_IMC,ESTADO_NUTRI,TARGET,PERC_GORDURA
0,47,1,1,114.5,178.0,36.14,Obesidade Grave,4,19.47
1,62,6,0,65.0,148.0,29.67,Risco/Sobrepeso,2,2.80
2,29,1,0,132.0,168.0,46.77,Obesidade Grave,4,51.72
3,1,4,0,13.3,90.0,16.42,Eutrofia,1,14.53
4,43,1,0,54.0,152.0,23.37,Eutrofia,1,6.59


In [3]:
# validando números de perc gordura

df['PERC_GORDURA'].describe()

count    1.533353e+06
mean     1.085482e+01
std      1.443678e+01
min     -5.174000e+01
25%      2.700000e+00
50%      1.145000e+01
75%      1.904000e+01
max      3.303200e+02
Name: PERC_GORDURA, dtype: float64

In [9]:
df[(df['PERC_GORDURA'] < 0) & (df['NU_IDADE_ANO'] > 15)].head(10).to_csv('validation_sample.csv')

In [12]:
df[(df['PERC_GORDURA'] < 0)]['NU_IDADE_ANO'].unique()

array([ 74,  55,  72,  65,  59,  61,  67,  60,  64,  52,  83,  69,  54,
        68,  82,  70,  62,  66,  88,  58,  75,  78,  73,  77,  57,  49,
        71,  85,  53,  76,  86,  63,  79,  81,  50,  48,  47,  51,  84,
        56,   4,  80,  87,  40,  90,  41,  89,  91,  46,   2,   0,  43,
        92,  36,  44,  95,   3,  45,  96,   1,  42,  93,   7,  34,  99,
         9,  35,   5,   6,  98,  38,  39,  10,  37,  94,   8, 106,  97,
       102, 100,  20,  16,  17,  33,  27,  14,  12,  11,  22,  21,  28,
        31,  29, 101,  13, 103,  15,  26,  32,  25, 104, 109,  24,  18,
        19,  30, 108, 105,  23, 107])

In [ ]:
age_col: str = "NU_IDADE_ANO"
imc_col: str = "DS_IMC"
sex_col: str = "SG_SEXO"
children_mask = df[age_col] <= 15
adults_mask = df[age_col] > 15

def calc_pg_adulto(imc: float,age: int, sex: int):
        print(f'Calculando pg age adulto: imc {imc}, age {age}, sex {sex}')
        return 1.51 * imc - 0.70 * age - 3.6 * sex + 1.4
def calc_pg_infantil(imc: float,age: int, sex: int):
        print(f'Calculando pg infantil: imc {imc}, age {age}, sex {sex}')
        return 1.20 * imc + 0.23 * age - 10.8 * sex - 5.4

In [6]:
calc_pg_adulto(df.loc[9,'DS_IMC'],df.loc[9,'NU_IDADE_ANO'],df.loc[9,'SG_SEXO'])
calc_pg_infantil(df.loc[9,'DS_IMC'],df.loc[9,'NU_IDADE_ANO'],df.loc[9,'SG_SEXO'])

Calculando pg age adulto: imc 33.2, age 74, sex 0
Calculando pg infantil: imc 33.2, age 74, sex 0


-0.26799999999999224

In [7]:
len(df.loc[adults_mask])

1059233

In [8]:
df.loc[adults_mask,'NU_IDADE_ANO'].min()

16

In [25]:
df.loc[adults_mask].head(15)

,NU_IDADE_ANO,DS_FASE_VIDA,SG_SEXO,NU_PESO,NU_ALTURA,DS_IMC,ESTADO_NUTRI,TARGET,PERC_GORDURA
0,47,1,1,114.5,178.0,36.14,Obesidade Grave,4,19.47
1,62,6,0,65.0,148.0,29.67,Risco/Sobrepeso,2,2.80
2,29,1,0,132.0,168.0,46.77,Obesidade Grave,4,51.72
4,43,1,0,54.0,152.0,23.37,Eutrofia,1,6.59
5,24,1,1,64.0,167.0,22.95,Eutrofia,1,15.65
6,25,1,0,49.0,157.0,19.88,Eutrofia,1,13.92
7,40,1,0,71.5,150.0,31.78,Obesidade,3,21.39
8,21,1,0,50.0,165.0,18.37,Baixo peso,0,14.44
9,74,6,0,85.0,160.0,33.20,Risco/Sobrepeso,2,-0.27
13,52,1,0,59.0,150.0,26.22,Risco/Sobrepeso,2,4.59


In [3]:
import joblib

In [4]:
models = joblib.load("models/artifacts/best_model.joblib")
print(models)

Pipeline(steps=[('clf',
                 RandomForestClassifier(max_depth=14, min_samples_split=3,
                                        n_estimators=312, n_jobs=-1,
                                        random_state=42))])


In [13]:
df_predictions = pd.read_csv('models/artifacts/best_model_predictions.csv')

In [14]:
df_predictions.head(2)

,NU_IDADE_ANO,DS_FASE_VIDA,SG_SEXO,NU_PESO,NU_ALTURA,DS_IMC,ESTADO_NUTRI,TARGET,PERC_GORDURA,Prediction
0,47,1,1,114.5,178.0,36.14,Obesidade Grave,4,19.47,Obesidade Grave
1,62,6,0,65.0,148.0,29.67,Risco/Sobrepeso,2,2.80,Risco/Sobrepeso


In [15]:
#Resposta Q1
df_predictions['Prediction'].value_counts()

Prediction
Eutrofia           630555
Risco/Sobrepeso    556349
Obesidade          178342
Obesidade Grave    110851
Baixo peso          57256
Name: count, dtype: int64

In [16]:
#Resposta Q2
df_predictions['NU_IDADE_ANO'].mean()

34.49353280034017

In [17]:
#Resposta Q3
len(df_predictions[df_predictions['Prediction'] == 'Obesidade Grave'])

110851

In [18]:
#Resposta Q4
len(df_predictions[df_predictions['NU_IDADE_ANO'] > 60])

288311